In [160]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

In [161]:
df = pd.read_csv("ventas_ecommerce_limpio.csv")

In [162]:
df.head()

,id_venta,fecha,cliente,producto,categoria,cantidad,precio_unitario,metodo_pago,ciudad,total_venta
0,1001,2026-07-01,Ana Lopez,Mouse,Accesorios,1,250.0,Efectivo,Cuernavaca,250.0
1,1002,2026-07-01,Luis Perez,Teclado,Accesorios,1,650.0,Tarjeta,Jiutepec,650.0
2,1003,2026-07-02,Sofia Ruiz,Audifonos,Accesorios,1,900.0,Tarjeta,Temixco,900.0
3,1004,2026-07-02,Pedro Mata,Webcam,Accesorios,1,800.0,Efectivo,Cuernavaca,800.0
4,1005,2026-07-03,Laura Diaz,Cable HDMI,Accesorios,2,180.0,Efectivo,Jiutepec,360.0


In [163]:
df.shape

(60, 10)

In [164]:
df.isnull().sum()

id_venta           0
fecha              0
cliente            0
producto           0
categoria          0
cantidad           0
precio_unitario    0
metodo_pago        0
ciudad             0
total_venta        0
dtype: int64

In [165]:
df.columns

Index(['id_venta', 'fecha', 'cliente', 'producto', 'categoria', 'cantidad',
       'precio_unitario', 'metodo_pago', 'ciudad', 'total_venta'],
      dtype='object')

In [166]:
df["total_calculado"] = df["cantidad"] * df["precio_unitario"]

In [167]:
df[df["total_venta"] != df["total_calculado"]]

,id_venta,fecha,cliente,producto,categoria,cantidad,precio_unitario,metodo_pago,ciudad,total_venta,total_calculado


In [168]:
# No se encontraron inconsistencias, no hay nulos, total venta esta correctamente bien calculado, y el data frame tiene 60 filas y 10 columnas

In [169]:
df["venta_alta"] = df["total_venta"].apply(lambda x: 1 if x >= 1000 else 0)

In [170]:
df["venta_alta"].value_counts()

venta_alta
1    40
0    20
Name: count, dtype: int64

In [171]:
# Por que venta_alta es la variable objetivo?
    # Porque es la varible que queremos predecir y por ello es el resultado que debemos pasarle al modelo para que encuentre patrones en la variable X que lo lleven a ese objetivo

In [172]:
X = df[["cantidad", "precio_unitario", "categoria", "metodo_pago", "ciudad"]]
y = df["venta_alta"]

In [173]:
X = pd.get_dummies(X)

In [174]:
columnas_modelo = X.columns.tolist()

In [175]:
X.head()

,cantidad,precio_unitario,categoria_Accesorios,categoria_Electronica,categoria_Muebles,metodo_pago_Efectivo,metodo_pago_Tarjeta,metodo_pago_Transferencia,ciudad_Cuernavaca,ciudad_Emiliano Zapata,ciudad_Jiutepec,ciudad_Temixco
0,1,250.0,True,False,False,True,False,False,True,False,False,False
1,1,650.0,True,False,False,False,True,False,False,False,True,False
2,1,900.0,True,False,False,False,True,False,False,False,False,True
3,1,800.0,True,False,False,True,False,False,True,False,False,False
4,2,180.0,True,False,False,True,False,False,False,False,True,False


In [176]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [177]:
modelo = DecisionTreeClassifier(random_state=42)
modelo.fit(X_train, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [178]:
predicciones = modelo.predict(X_test)

In [179]:
exactitud = accuracy_score(y_test, predicciones)
print("Exactitud:", exactitud)

Exactitud: 1.0


In [180]:
matriz = confusion_matrix(y_test, predicciones)
print(matriz)

[[4 0]
 [0 8]]


In [181]:
resultados_prueba = pd.DataFrame({
    "valor_real": y_test,
    "prediccion": predicciones
})

In [182]:
resultados_prueba["coincide"] = resultados_prueba["valor_real"] == resultados_prueba["prediccion"]


In [183]:
resultados_prueba["coincide"].value_counts()


coincide
True    12
Name: count, dtype: int64

In [184]:
aciertos = resultados_prueba[resultados_prueba["coincide"] == True]
errores = resultados_prueba[resultados_prueba["coincide"] == False]

len(aciertos)
len(errores)

0

In [185]:
# Preguntas obligatorias:

# 1. Cual fue la exactitud?
    # Tuvo una exactitud de 1, es decir un 100% lo cual indica que predijo a la perfeccion aunque no es muy realista ya que fueron pocos datos
# 2. Cuantos aciertos tuvo el modelo?
    # 12 aciertos
# 3. Cuantos errores tuvo el modelo?
    # No tuvo ningun error
# 4. Que indica la matriz de confusion?
    # Indica que hubo 4 valores positivos correctamente bien predichos, y 8 valores negativos correctamente bien predichos
# 5. Una buena exactitud significa que el modelo ya es perfecto? Explica.
    # No necesariamente, sobre todo en casos como este con tan pocos datos de entrenamiento y tan pocos datos de prueba pudo haberse debido a una mera casualidad mas que a la verdadera perfeccion del modelo

In [186]:
joblib.dump(modelo, "modelo_examen_venta_alta.pkl")

['modelo_examen_venta_alta.pkl']

In [187]:
joblib.dump(columnas_modelo, "columnas_examen_modelo.pkl")

['columnas_examen_modelo.pkl']

In [188]:
# Preguntas obligatorias:

# 1. Para que sirve guardar el modelo?
    # Para no tener que estarlo entrenando una y otra vez cada que queremos usarlo.
# 2. Para que sirve guardar las columnas del entrenamiento?
    # Para que el modelo no se pierda cuado tenga que volver a trabajar con esos datos
# 3. Que problema puede aparecer si no guardas las columnas?
    # Que si a la proxima que necesita trabajar con esas columnas y ocurre algo en ellas o estan mal podria producir erroes que hagan que de malas predicciones.

In [189]:
nuevas = pd.DataFrame({
    "id_venta": [2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010],
    "fecha": ["2026-08-07", "2026-08-07", "2026-08-07", "2026-08-07", "2026-08-07", "2026-08-07", "2026-08-07", "2026-08-07", "2026-08-07", "2026-08-07"],
    "cliente": ["Cliente A", "Cliente B", "Cliente C", "Cliete D", "Cliente E", "Cliente F", "Cliente G", "Cliente A", "Cliente B", "Cliente C"],
    "producto": ["Mouse", "Laptop", "Audifonos", "Lavadora", "Mousepad", "RAM", "Lentes VR", "Monitor", "Mouse", "C4"],
    "categoria": ["Accesorios", "Electronica", "Accesorios", "Electronica", "Accesorios", "Electronica", "Accesorios", "Electronica", "Accesorios", "Armamento"],
    "cantidad": [1, 1, 4,2,6,8,9,2,4,5],
    "precio_unitario": [500, 12000, 4554,7463,545,2443,950,1200,263,43],
    "metodo_pago": ["Efectivo", "Tarjeta", "Efectivo", "Tarjeta",  "Efectivo", "Tarjeta",  "Efectivo", "Tarjeta",  "Efectivo", "Tarjeta"],
    "ciudad": ["Cuernavaca", "Paris", "Cuernavaca", "Jiutepec",  "Cuernavaca", "Jiutepec",  "Cuernavaca", "Jiutepec",  "Cuernavaca", "Jiutepec", ]
})

In [190]:
nuevas.to_csv("examen_ventas_nuevas.csv", index=False)

In [191]:
modelo_cargado = joblib.load("modelo_examen_venta_alta.pkl")
columnas_modelo = joblib.load("columnas_examen_modelo.pkl")

In [192]:
nuevas = pd.read_csv("examen_ventas_nuevas.csv")

In [193]:
X_nuevas = nuevas[["cantidad", "precio_unitario", "categoria", "metodo_pago", "ciudad"]]
X_nuevas = pd.get_dummies(X_nuevas)

In [194]:
X_nuevas = X_nuevas.reindex(columns=columnas_modelo, fill_value=0)

In [195]:
nuevas["prediccion_venta_alta"] = modelo_cargado.predict(X_nuevas)

In [196]:
nuevas["interpretacion_prediccion"] = nuevas["prediccion_venta_alta"].map({
    0: "Venta no alta",
    1: "Venta alta"
})

In [197]:
nuevas["interpretacion_prediccion"]

0    Venta no alta
1       Venta alta
2    Venta no alta
3       Venta alta
4    Venta no alta
5       Venta alta
6    Venta no alta
7       Venta alta
8    Venta no alta
9       Venta alta
Name: interpretacion_prediccion, dtype: object

In [198]:
nuevas["total_estimado"] = nuevas["cantidad"] * nuevas["precio_unitario"]

In [199]:
nuevas["venta_alta_real_estimada"] = nuevas["total_estimado"].apply(lambda x: 1 if x >= 1000 else 0)

In [200]:
nuevas["venta_alta_real_estimada"]

0    0
1    1
2    1
3    1
4    1
5    1
6    1
7    1
8    1
9    0
Name: venta_alta_real_estimada, dtype: int64

In [201]:
nuevas["coincide"] = nuevas["prediccion_venta_alta"] == nuevas["venta_alta_real_estimada"]

In [202]:
nuevas["coincide"]

0     True
1     True
2    False
3     True
4    False
5     True
6    False
7     True
8    False
9    False
Name: coincide, dtype: bool

In [203]:
nuevas["coincide"].value_counts()

coincide
True     5
False    5
Name: count, dtype: int64

In [204]:
aciertos = nuevas[nuevas["coincide"] == True]
len(aciertos)

5

In [205]:
errores = nuevas[nuevas["coincide"] == False]
len(errores)

5

In [206]:
nuevas["distancia_a_1000"] = (nuevas["total_estimado"] - 1000).abs()

In [207]:
cerca_limite = nuevas[nuevas["distancia_a_1000"] <= 200]

In [208]:
nuevas.to_csv("examen_predicciones.csv", index=False)

In [209]:
nuevas

,id_venta,fecha,cliente,producto,categoria,cantidad,precio_unitario,metodo_pago,ciudad,prediccion_venta_alta,interpretacion_prediccion,total_estimado,venta_alta_real_estimada,coincide,distancia_a_1000
0,2001,2026-08-07,Cliente A,Mouse,Accesorios,1,500,Efectivo,Cuernavaca,0,Venta no alta,500,0,True,500
1,2002,2026-08-07,Cliente B,Laptop,Electronica,1,12000,Tarjeta,Paris,1,Venta alta,12000,1,True,11000
2,2003,2026-08-07,Cliente C,Audifonos,Accesorios,4,4554,Efectivo,Cuernavaca,0,Venta no alta,18216,1,False,17216
3,2004,2026-08-07,Cliete D,Lavadora,Electronica,2,7463,Tarjeta,Jiutepec,1,Venta alta,14926,1,True,13926
4,2005,2026-08-07,Cliente E,Mousepad,Accesorios,6,545,Efectivo,Cuernavaca,0,Venta no alta,3270,1,False,2270
5,2006,2026-08-07,Cliente F,RAM,Electronica,8,2443,Tarjeta,Jiutepec,1,Venta alta,19544,1,True,18544
6,2007,2026-08-07,Cliente G,Lentes VR,Accesorios,9,950,Efectivo,Cuernavaca,0,Venta no alta,8550,1,False,7550
7,2008,2026-08-07,Cliente A,Monitor,Electronica,2,1200,Tarjeta,Jiutepec,1,Venta alta,2400,1,True,1400
8,2009,2026-08-07,Cliente B,Mouse,Accesorios,4,263,Efectivo,Cuernavaca,0,Venta no alta,1052,1,False,52
9,2010,2026-08-07,Cliente C,C4,Armamento,5,43,Tarjeta,Jiutepec,1,Venta alta,215,0,False,785


In [210]:
# Preguntas obligatorias:

# 1. Cuantas ventas nuevas evaluaste?
    #Evalue las 10 nuevas ventas que agregue porque el modelo ya estaba previamente entrenado
# 2. Cuantas fueron predichas como venta alta?
    # El modelo predijo que 5/10 eran ventas altas
# 3. Cuantas fueron predichas como venta no alta?
    # El modelo predijo 5/10 ventas como no altas
# 4. Cuantas coincidieron con la regla manual?
    #  5
# 5. Cuantas no coincidieron?
    # 5
# 6. Que ventas no coincidieron?
    # Las ventas que no coincidieron fueron: 
        # 4 Audifonos por distancia a 1000 de 17216
        # 6 MousePad, distancia: 2270
        # 9 LentesVR, distancia: 7550
        # 4 Mouse, distancia 52
        # C4, distancia 785
# 7. Los errores estuvieron cerca del limite de 1000?
    #Aunque algunos errores si lo estuvieron no fue el caso para todos, de hecho varios estaban bastante alejados
# 8. Que paso con la categoria nueva?
    #Fallo, podria deberse a que fallo por no haber sido entrenada con esa informacion (o una casualidad) ya que con exactitud del 50% practicamente eligio al azar
# 9. Que paso con la ciudad nueva?
    #EL modelo lo predijo correctamente a pesar de la ciudad nueva

In [211]:
# Nombre: Gaspar Andres Garcia Quiroz
# Grupo: 9A
# Materia: Extraccion de Conocimiento de Base de Datos
# Exactitud obtenida: 0.5
# Ventas nuevas evaluadas: 10
# Coincidencias: 5 
# Errores: 5 
# Conclusion breve: Mi modelo obtuvo una exactitud del 50% lo que hace parecer que el modelo no aprendio nada en una primera instancia, pero la realidad es que un modelo requiere de miles de datos para poder realmente aprender, ademas de que realmente con esta imprecision de mi modelo no pude apreciar si realmente existia un cambio entre el haber añadido la nueva categoria y la nueva ciudad